# 🌟 Astra Core — AI Agent (No API Key Needed)

A tool-using AI agent that **runs completely free: no API key, no `pip install`, no account.**
Just click **Run All** and chat with it in the last cell.

**What Astra Core can do**
| Skill | Try saying |
|---|---|
| 🧮 Calculator | `what is 25 * 4 + 10`, `20% of 150`, `square root of 144` |
| 🕒 Time & date | `what time is it`, `time in Tokyo` |
| 📚 Wikipedia | `who is APJ Abdul Kalam`, `tell me about black holes` |
| 🌦️ Weather | `weather in Coimbatore` |
| 📝 Notes (memory) | `remember buy milk`, `show notes` |
| 🙋 Remembers you | `my name is Ravi`, `what is my name` |
| 🎲 Fun | `tell me a joke`, `flip a coin`, `roll a dice` |

Only Wikipedia and Weather need internet. Everything else works offline.

**How it works:** the agent reads your message → *plans* which tool to use → runs the tool → replies. Same agent loop as LLM-based agents, but with a built-in rule-based planner instead of a paid model.

## 1. Setup (standard library only)

In [ ]:
import re, ast, json, os, random, operator
import urllib.request, urllib.parse
from datetime import datetime

NOTES_FILE = "astra_notes.json"
HEADERS = {"User-Agent": "AstraCore/1.0 (educational project)"}

def http_json(url, timeout=10):
    req = urllib.request.Request(url, headers=HEADERS)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))

NO_INTERNET = "I couldn't reach the internet right now. Please check your connection and try again."
print("✅ Setup done")

## 2. Tools
Each tool is a normal Python function that takes simple inputs and returns text.

In [ ]:
# ---------- 🧮 Calculator (safe: no eval) ----------
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.FloorDiv: operator.floordiv, ast.USub: operator.neg, ast.UAdd: operator.pos,
}

def _eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        left, right = _eval(node.left), _eval(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 1000:
            raise ValueError("Exponent too large")
        return _OPS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("Unsupported expression")

def calculator(expression):
    try:
        result = _eval(ast.parse(expression, mode="eval").body)
        if isinstance(result, float):
            result = int(result) if result.is_integer() else round(result, 6)
        return f"{expression.strip()} = {result}"
    except ZeroDivisionError:
        return "I can't divide by zero."
    except Exception:
        return "Sorry, I couldn't work out that calculation."

# ---------- 🕒 Time ----------
CITY_TZ = {
    "india": "Asia/Kolkata", "chennai": "Asia/Kolkata", "coimbatore": "Asia/Kolkata",
    "delhi": "Asia/Kolkata", "mumbai": "Asia/Kolkata", "bangalore": "Asia/Kolkata",
    "kolkata": "Asia/Kolkata", "hyderabad": "Asia/Kolkata",
    "london": "Europe/London", "paris": "Europe/Paris", "berlin": "Europe/Berlin",
    "new york": "America/New_York", "los angeles": "America/Los_Angeles",
    "tokyo": "Asia/Tokyo", "dubai": "Asia/Dubai", "singapore": "Asia/Singapore",
    "sydney": "Australia/Sydney", "utc": "UTC",
}

def get_current_time(place=None):
    fmt = "%A, %d %B %Y, %I:%M %p"
    if not place:
        return "Your local time is " + datetime.now().strftime(fmt)
    try:
        from zoneinfo import ZoneInfo
        tz = CITY_TZ.get(place.lower().strip(), place.strip())
        return f"Time in {place.title()}: " + datetime.now(ZoneInfo(tz)).strftime(fmt)
    except Exception:
        return ("I don't know that place. Try: " + ", ".join(sorted(CITY_TZ)) +
                ". (On Windows you may need: pip install tzdata)")

# ---------- 📚 Wikipedia (free, no key) ----------
def search_wikipedia(topic):
    try:
        q = urllib.parse.quote(topic)
        found = http_json(f"https://en.wikipedia.org/w/api.php?action=opensearch&search={q}&limit=1&format=json")
        if not found[1]:
            return f"I couldn't find anything on Wikipedia for '{topic}'."
        title = found[1][0]
        page = http_json("https://en.wikipedia.org/api/rest_v1/page/summary/" +
                         urllib.parse.quote(title.replace(" ", "_")))
        return f"📚 {page.get('title', title)}: {page.get('extract', 'No summary available.')}"
    except Exception:
        return NO_INTERNET

# ---------- 🌦️ Weather (Open-Meteo, free, no key) ----------
WEATHER_CODES = {
    0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast",
    45: "foggy", 48: "foggy", 51: "light drizzle", 53: "drizzle", 55: "heavy drizzle",
    61: "light rain", 63: "rain", 65: "heavy rain", 80: "rain showers", 81: "rain showers",
    82: "violent rain showers", 95: "thunderstorm", 96: "thunderstorm with hail", 99: "thunderstorm with hail",
}

def get_weather(city):
    try:
        geo = http_json("https://geocoding-api.open-meteo.com/v1/search?count=1&name=" + urllib.parse.quote(city))
        if not geo.get("results"):
            return f"I couldn't find a place called '{city}'."
        g = geo["results"][0]
        w = http_json("https://api.open-meteo.com/v1/forecast?latitude=%s&longitude=%s"
                      "&current=temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code"
                      % (g["latitude"], g["longitude"]))["current"]
        sky = WEATHER_CODES.get(w["weather_code"], "unknown conditions")
        return (f"🌦️ {g['name']}, {g.get('country', '')}: {w['temperature_2m']}°C, {sky}, "
                f"humidity {w['relative_humidity_2m']}%, wind {w['wind_speed_10m']} km/h.")
    except Exception:
        return NO_INTERNET

# ---------- 📝 Notes (persistent memory) ----------
def _load_notes():
    if os.path.exists(NOTES_FILE):
        with open(NOTES_FILE) as f:
            return json.load(f)
    return []

def save_note(note):
    notes = _load_notes()
    notes.append({"time": datetime.now().isoformat(timespec="seconds"), "note": note})
    with open(NOTES_FILE, "w") as f:
        json.dump(notes, f, indent=2)
    return f"📝 Saved: {note}"

def read_notes():
    notes = _load_notes()
    if not notes:
        return "You have no saved notes yet."
    return "📝 Your notes:\n" + "\n".join(f"  {i}. {n['note']}  ({n['time']})" for i, n in enumerate(notes, 1))

def clear_notes():
    if os.path.exists(NOTES_FILE):
        os.remove(NOTES_FILE)
    return "🗑️ All notes cleared."

# ---------- 🎲 Fun ----------
JOKES = [
    "Why do programmers prefer dark mode? Because light attracts bugs.",
    "Why did the Python developer get glasses? Because they couldn't C.",
    "There are 10 kinds of people: those who understand binary and those who don't.",
    "A SQL query walks into a bar, walks up to two tables and asks: 'Can I join you?'",
    "I would tell you a UDP joke, but you might not get it.",
]
def tell_joke():  return "😄 " + random.choice(JOKES)
def flip_coin():  return "🪙 " + random.choice(["Heads", "Tails"])
def roll_dice():  return f"🎲 You rolled a {random.randint(1, 6)}"

print("✅ Tools ready")

## 3. The agent (planner + tool loop)
The planner turns your sentence into a tool call. You can add your own skills by adding a function above and one rule in `plan()`.

In [ ]:
def extract_math(text):
    """Pull a math expression out of a sentence, or return None."""
    t = text.lower()
    m = re.search(r"(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)", t)
    if m:
        return f"{m.group(1)}/100*{m.group(2)}"
    m = re.search(r"(?:square root|sqrt)(?: of)?\s*(\d+(?:\.\d+)?)", t)
    if m:
        return f"{m.group(1)}**0.5"
    for pattern, symbol in [(r"\bplus\b", "+"), (r"\bminus\b", "-"),
                            (r"\b(?:multiplied by|times)\b", "*"),
                            (r"\bdivided by\b", "/"), (r"\bto the power(?: of)?\b", "**"),
                            (r"\bmod\b", "%")]:
        t = re.sub(pattern, symbol, t)
    t = re.sub(r"(?<=\d)\s*x\s*(?=\d)", "*", t).replace("^", "**")
    m = re.search(r"\(*\s*\d[\d\s+\-*/().%]*\d\s*\)*", t)
    if m and re.search(r"[+\-*/%]", m.group(0)):
        return m.group(0).strip()
    return None


class AstraCore:
    def __init__(self, verbose=True):
        self.verbose = verbose
        self.memory = {"name": None}   # remembers things about you this session
        self.history = []              # (user, astra) conversation log

    # ----- Step 1: PLAN -> choose a tool -----
    def plan(self, text):
        t = text.strip()
        low = t.lower().rstrip("?!. ")

        m = re.search(r"\bmy name is\s+([A-Za-z][A-Za-z .'-]*)", t, re.I)
        if m:
            return "set_name", {"name": m.group(1).strip().title()}
        if re.search(r"\bwhat(?:'s| is) my name\b|\bwho am i\b", low):
            return "get_name", {}

        if re.search(r"\b(clear|delete|erase)\b.*\bnotes?\b", low):
            return "clear_notes", {}
        m = re.match(r"^(?:please\s+)?(?:remember|note down|note|save note|save)(?: that)?\s+(.+)", t, re.I)
        if m:
            return "save_note", {"note": m.group(1).strip()}
        if re.search(r"\b(show|read|list|view|see|what)\b.*\b(notes?|remembered)\b", low):
            return "read_notes", {}

        expr = extract_math(low)
        if expr:
            return "calculator", {"expression": expr}

        m = re.search(r"\bweather(?:\s+(?:in|at|for|of))?\s+([a-z ,.-]+)$", low)
        if m:
            return "get_weather", {"city": m.group(1).strip()}

        if re.search(r"\b(what(?:'s| is)? (?:the )?(?:current )?(?:time|date)|current (?:time|date)|time now|today'?s date|what day)\b", low) \
                or low in {"time", "date"} or re.match(r"^time (?:in|at) ", low):
            m = re.search(r"\b(?:in|at|for)\s+([a-z/_ ]+)$", low)
            return "get_current_time", {"place": m.group(1).strip() if m else None}

        if "joke" in low:
            return "tell_joke", {}
        if re.search(r"\b(flip|toss)\b.*\bcoin\b", low):
            return "flip_coin", {}
        if re.search(r"\b(roll|throw)\b.*\bdic?e\b|\bdice\b", low):
            return "roll_dice", {}

        if re.match(r"^(hi|hello|hey|vanakkam|good (morning|afternoon|evening))\b", low):
            return "greet", {}
        if re.search(r"\b(thanks|thank you|nandri)\b", low):
            return "thanks", {}
        if re.search(r"\bhelp\b|what can you do", low):
            return "help", {}

        m = re.match(r"^(?:who is|who was|who are|what is|what are|what was|tell me about|"
                     r"search for|search|define|explain|wiki|information about)\s+(.+)$", low)
        if m:
            topic = re.sub(r"^(?:a|an|the)\s+", "", m.group(1)).strip()
            return "search_wikipedia", {"topic": topic}

        return "fallback", {}

    # ----- Step 2: ACT -> run the tool -----
    def act(self, tool, args):
        if tool == "set_name":
            self.memory["name"] = args["name"]
            return f"Nice to meet you, {args['name']}! I'll remember your name."
        if tool == "get_name":
            n = self.memory["name"]
            return f"Your name is {n}." if n else "I don't know your name yet. Say: my name is ..."
        if tool == "greet":
            n = self.memory["name"]
            return f"Hello{', ' + n if n else ''}! 👋 I'm Astra Core. Type 'help' to see what I can do."
        if tool == "thanks":
            return "You're welcome! 😊"
        if tool == "help":
            return HELP_TEXT
        if tool == "fallback":
            return ("I'm not sure how to do that yet 🤔. Try a calculation, 'time in Tokyo', "
                    "'weather in Chennai', 'who is Albert Einstein', or 'remember ...'. Type 'help' for more.")
        return globals()[tool](**args)

    # ----- Step 3: RESPOND (the loop) -----
    def run(self, text):
        tool, args = self.plan(text)
        if self.verbose and tool not in {"greet", "thanks", "help", "fallback", "set_name", "get_name"}:
            print(f"  🔧 using tool: {tool} {args}")
        reply = self.act(tool, args)
        self.history.append((text, reply))
        return reply

    def reset(self):
        self.memory = {"name": None}
        self.history = []


HELP_TEXT = """Here's what I can do:
  🧮 Math:       25 * 4 + 10 | 20% of 150 | square root of 144
  🕒 Time:       what time is it | time in Tokyo
  📚 Wikipedia:  who is APJ Abdul Kalam | tell me about black holes
  🌦️ Weather:    weather in Coimbatore
  📝 Notes:      remember buy milk | show notes | clear notes
  🙋 About you:  my name is Ravi | what is my name
  🎲 Fun:        tell me a joke | flip a coin | roll a dice
  Type 'reset' to clear memory, 'exit' to quit."""

agent = AstraCore()
print("✅ Astra Core is ready!")

## 4. Try it out

In [ ]:
for q in ["hello", "my name is Ravi", "what is my name", "what is 25 * 4 + 10",
          "20% of 150", "square root of 144", "what time is it",
          "remember buy milk tomorrow", "show notes", "tell me a joke", "flip a coin"]:
    print("You:  ", q)
    print("Astra:", agent.run(q), "\n")

### Internet skills (Wikipedia + Weather)

In [ ]:
for q in ["who is APJ Abdul Kalam", "weather in Coimbatore"]:
    print("You:  ", q)
    print("Astra:", agent.run(q), "\n")

## 5. 💬 Chat with Astra Core
Run this cell and start chatting. Type `help` to see commands, `exit` to stop.

In [ ]:
print("🌟 Astra Core — type 'help' for commands, 'exit' to quit.\n")
while True:
    user = input("You: ").strip()
    if not user:
        continue
    if user.lower() in {"exit", "quit", "bye"}:
        print("Astra: Goodbye! 👋")
        break
    if user.lower() == "reset":
        agent.reset()
        print("Astra: Memory cleared.\n")
        continue
    print("Astra:", agent.run(user), "\n")

## 6. Want to level up?
- Add your own tool: write a function, then add one rule in `plan()`
- Connect a real LLM later (Anthropic API or a free local model with **Ollama**) and keep these tools as its toolbox
- Wrap it in a **Gradio** or **Streamlit** chat UI

*Astra Core — runs free, anywhere Jupyter runs.*